# 06 · GroupBy, aggregate functions and distinct counts

**Scenario:** Build per-origin trip statistics and compare row counts with distinct fingerprints.

**Source:** Databricks-provided `samples.nyctaxi.trips` only. File formats, nested structures, replays and histories in this lab are derived from that asset. No hand-created record dataset, third-party download, pandas or generated replacement data is used.

**Prerequisites:** Run notebook 00 first. Each notebook is independent; run its cells in order. Mode: **dataframe**. 

**Learning-size sample:** `max_rows` defaults to 2,000. Sorting/limiting caps result size, not the underlying scan cost. Assertions use relationships instead of hardcoded dataset counts because sample assets may change.


## Step 1 — Load and inspect the Databricks asset

Read the built-in sample and normalize column names. A stable content fingerprint helps with replay examples; it is not a guaranteed real-world trip key. The original `samples` asset is always read-only.


In [ ]:
from pyspark.sql import functions as F, types as T, Window
dbutils.widgets.text('max_rows','2000')
MAX_ROWS=int(dbutils.widgets.get('max_rows'))
assert 100<=MAX_ROWS<=20000, 'Use 100–20000 rows for these bounded learning labs.'
SOURCE_ASSET='samples.nyctaxi.trips'
try:
    asset=(spark.read.table(SOURCE_ASSET)
        .select('tpep_pickup_datetime','tpep_dropoff_datetime','trip_distance',
                'fare_amount','pickup_zip','dropoff_zip')
        .orderBy('tpep_pickup_datetime','tpep_dropoff_datetime','pickup_zip',
                 'dropoff_zip','trip_distance','fare_amount')
        .limit(MAX_ROWS))
    assert asset.limit(1).count()==1, 'Databricks sample asset is empty.'
except Exception as exc:
    raise RuntimeError('This course requires the Databricks-provided samples.nyctaxi.trips asset. '
        'Check Unity Catalog/sample access on the attached compute. No synthetic or external fallback is used.') from exc
df=(asset.select(F.col('tpep_pickup_datetime').cast('timestamp').alias('pickup_ts'),
    F.col('tpep_dropoff_datetime').cast('timestamp').alias('dropoff_ts'),
    F.col('trip_distance').cast('double').alias('distance'),
    F.col('fare_amount').cast('double').alias('fare'),
    F.col('pickup_zip').cast('string').alias('pickup_zip'),
    F.col('dropoff_zip').cast('string').alias('dropoff_zip')))
df=df.withColumn('trip_id',F.sha2(F.to_json(F.struct(*[F.col(c) for c in df.columns]),
    options={'ignoreNullFields':'false'}),256))
df=df.withColumn('pickup_date',F.to_date('pickup_ts'))
df=df.withColumn('duration_sec',F.unix_timestamp('dropoff_ts')-F.unix_timestamp('pickup_ts'))
print('Source:',SOURCE_ASSET,'sample limit:',MAX_ROWS)
# trip_id is a content fingerprint, NOT an official source/business trip identifier.
# Identical source rows intentionally have identical fingerprints.
df.show(5,truncate=False)


## Step 2 — Aggregate once for multiple metrics

Put related measures in one aggregation so Spark can compute them together.


In [ ]:
summary=df.groupBy('pickup_zip').agg(F.count('*').alias('trips'),F.sum('fare').alias('fare_total'),
    F.avg('distance').alias('mean_distance'),F.countDistinct('dropoff_zip').alias('destinations'))
summary.orderBy(F.desc('trips')).show(10)
assert summary.agg(F.sum('trips')).first()[0]==df.count()


**Expected check:** The sum of per-origin trip counts equals the input row count, including the NULL-origin group.


## Step 3 — Compute weighted averages correctly

Average of group averages is generally not the overall average. Weight by non-null observations for the metric.


In [ ]:
parts=df.groupBy('pickup_zip').agg(F.sum('fare').alias('total'),F.count('fare').alias('n'))
weighted=parts.agg((F.sum('total')/F.sum('n')).alias('overall')).first()[0]
direct=df.agg(F.avg('fare')).first()[0]
assert abs(weighted-direct)<1e-6
print('Weighted overall mean:',weighted)


**Expected check:** The weighted mean equals the direct overall mean within numeric tolerance.


## Practice

Add average duration but exclude negative durations. Which count is the correct denominator?

## Solution guide

Use avg(when(duration_sec >= 0, duration_sec)). For manual weighting, count only non-null nonnegative duration values, not all trips.

## Production considerations

Keep source lineage and stable business keys. Measure on representative volumes, review nulls and time semantics, and make retries idempotent. Sample-data checks demonstrate mechanics; they do not establish production performance or exactly-once external writes.
